# Federated Learning from Scratch
### Replicating McMahan et al. 2017 — FedAvg Algorithm

**Core idea**: Train a shared model across many devices without ever moving the raw data to a server.

| What we build | Why |
|---|---|
| IID vs Non-IID data split | The paper's key challenge |
| FedSGD | Naive baseline |
| FedAvg | The paper's contribution |
| Convergence plots | Reproduce Figure 2 |

In [ ]:
import copy, random
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Reproducibility
torch.manual_seed(42)
random.seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

## Step 1 — Data: IID vs Non-IID

The paper's most important challenge: real phones have **non-IID data**.
Your phone knows your writing style, not the average of all humans.

- **IID**: shuffle all data, split equally → every client sees all 10 digits
- **Non-IID**: sort by label, assign 2 shards per client → most clients see only **2 digit classes**

In [ ]:
NUM_CLIENTS = 10  # small for speed; paper uses 100

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_data = datasets.MNIST('./data', train=True,  download=True, transform=transform)
test_data  = datasets.MNIST('./data', train=False, download=True, transform=transform)
test_loader = DataLoader(test_data, batch_size=512, shuffle=False)

print(f"Train: {len(train_data)} samples | Test: {len(test_data)} samples")


def partition_iid(dataset, num_clients):
    """Shuffle all indices, split equally across clients."""
    idx = list(range(len(dataset)))
    random.shuffle(idx)
    size = len(idx) // num_clients
    return {i: idx[i*size:(i+1)*size] for i in range(num_clients)}


def partition_non_iid(dataset, num_clients, shards_per_client=2):
    """
    Pathological split from the paper:
    1. Sort data by label
    2. Cut into (num_clients * shards_per_client) shards
    3. Give each client exactly shards_per_client shards
    → Most clients see only 1-2 digit classes
    """
    labels  = np.array(dataset.targets)
    sorted_idx = np.argsort(labels)

    num_shards = num_clients * shards_per_client
    shard_size = len(sorted_idx) // num_shards
    shards     = [sorted_idx[i*shard_size:(i+1)*shard_size] for i in range(num_shards)]

    shard_ids  = list(range(num_shards))
    random.shuffle(shard_ids)

    client_data = {}
    for i in range(num_clients):
        assigned = shard_ids[i*shards_per_client:(i+1)*shards_per_client]
        client_data[i] = np.concatenate([shards[s] for s in assigned]).tolist()
    return client_data


iid_part     = partition_iid(train_data, NUM_CLIENTS)
non_iid_part = partition_non_iid(train_data, NUM_CLIENTS)

# Show what each client sees
labels = np.array(train_data.targets)
print("\nClasses per client:")
print("  IID    :", [sorted(set(labels[iid_part[i]].tolist()))     for i in range(5)])
print("  Non-IID:", [sorted(set(labels[non_iid_part[i]].tolist())) for i in range(5)])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle("What each client sees — IID vs Non-IID", fontsize=13)

for ax, part, title in zip(axes, [iid_part, non_iid_part], ["IID", "Non-IID (pathological)"]):
    mat = np.zeros((NUM_CLIENTS, 10))
    for c in range(NUM_CLIENTS):
        for d in range(10):
            mat[c, d] = np.sum(labels[part[c]] == d)
    im = ax.imshow(mat, aspect='auto', cmap='YlOrRd')
    ax.set_title(title, fontsize=11)
    ax.set_xlabel("Digit class (0-9)")
    ax.set_ylabel("Client ID")
    ax.set_xticks(range(10))
    plt.colorbar(im, ax=ax, label="# samples")

plt.tight_layout()
plt.show()

print("Non-IID: bright rows = client has many of one digit, ZERO of others.")
print("This is why federated learning is hard!")

## Step 2 — Model Architecture

Simple 2-layer MLP (the paper's "2NN"):
- Input: 784 pixels
- Hidden: 200 units, ReLU
- Hidden: 200 units, ReLU  
- Output: 10 classes

Fast enough to see the convergence difference clearly.

In [ ]:
class MLP(nn.Module):
    """
    2NN from McMahan et al. Section 3.
    Two hidden layers, 200 units each, ReLU activations.
    ~199K parameters.
    """
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 200), nn.ReLU(),
            nn.Linear(200, 200),   nn.ReLU(),
            nn.Linear(200, 10)
        )
    def forward(self, x):
        return self.net(x)


# Sanity check
m = MLP()
dummy = torch.randn(4, 1, 28, 28)
assert m(dummy).shape == (4, 10)
n_params = sum(p.numel() for p in m.parameters())
print(f"Model: MLP | Parameters: {n_params:,}")
print("Shape test passed: (4,1,28,28) → (4,10) ✓")

## Step 3 — FedSGD (Baseline)

### Formula
$$w_{t+1} \leftarrow w_t - \eta \sum_{k=1}^{K} \frac{n_k}{n} \nabla F_k(w_t)$$

Each client computes **one gradient** on its local data.  
Server does a **weighted average of gradients**.  
This is just distributed SGD — very expensive in communication rounds.

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    crit = nn.CrossEntropyLoss(reduction='sum')
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            out = model(x)
            loss_sum += crit(out, y).item()
            correct  += out.argmax(1).eq(y).sum().item()
            total    += len(y)
    return loss_sum / total, 100.0 * correct / total


def get_flat(model):
    return torch.cat([p.data.view(-1) for p in model.parameters()])

def set_flat(model, flat):
    offset = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[offset:offset+n].view_as(p))
        offset += n


def fed_sgd(partition, num_rounds=30, C=0.3, lr=0.1):
    """
    FedSGD: each selected client computes one gradient (full local batch).
    Server applies weighted average of gradients.
    """
    model  = MLP().to(DEVICE)
    n_tot  = sum(len(partition[c]) for c in range(NUM_CLIENTS))
    m      = max(1, int(C * NUM_CLIENTS))
    hist   = {'round': [], 'acc': []}

    for rnd in range(1, num_rounds + 1):
        selected   = random.sample(range(NUM_CLIENTS), m)
        agg_grad   = None

        for cid in selected:
            idx    = partition[cid]
            loader = DataLoader(Subset(train_data, idx), batch_size=len(idx))
            x, y   = next(iter(loader))
            x, y   = x.to(DEVICE), y.to(DEVICE)

            cm = copy.deepcopy(model)
            cm.train()
            opt = optim.SGD(cm.parameters(), lr=lr)
            opt.zero_grad()
            nn.CrossEntropyLoss()(cm(x), y).backward()

            # Collect gradient, weight by n_k / n
            g = torch.cat([p.grad.view(-1) for p in cm.parameters()])
            w = len(idx) / n_tot
            agg_grad = w * g if agg_grad is None else agg_grad + w * g

        # Server step: w = w - lr * agg_gradient
        set_flat(model, get_flat(model) - lr * agg_grad)

        _, acc = evaluate(model, test_loader)
        hist['round'].append(rnd)
        hist['acc'].append(acc)
        if rnd % 5 == 0:
            print(f"  [FedSGD] Round {rnd:3d} | Acc: {acc:.1f}%")

    return hist


print("Running FedSGD...")
hist_sgd_iid     = fed_sgd(iid_part,     num_rounds=30, C=0.3, lr=0.1)
print("\nNon-IID:")
hist_sgd_noniid  = fed_sgd(non_iid_part, num_rounds=30, C=0.3, lr=0.1)

## Step 4 — FedAvg (The Paper's Algorithm 1)

### Key difference from FedSGD

| | FedSGD | FedAvg |
|---|---|---|
| What clients send | Gradient (1 step) | Updated weights (E epochs) |
| Server does | Averages gradients | Averages weights |
| Communication | 1 round = 1 step | 1 round = many steps |

### Server aggregation formula
$$w_{t+1} = \sum_{k \in S_t} \frac{n_k}{n} \cdot w_{t+1}^k$$

### Three control knobs
- **C** — fraction of clients per round
- **E** — local epochs per round ← the main knob
- **B** — local minibatch size

More E = more local compute = fewer communication rounds needed.

In [ ]:
def client_update(global_model, idx, E, B, lr):
    """
    ClientUpdate(k, w) from Algorithm 1:
    - Copy global model
    - Run E local epochs with minibatch size B
    - Return updated weights (never sends raw data)
    """
    model = copy.deepcopy(global_model).to(DEVICE)
    model.train()
    batch_size = len(idx) if B == 'inf' else B
    loader = DataLoader(Subset(train_data, idx), batch_size=batch_size, shuffle=True)
    opt  = optim.SGD(model.parameters(), lr=lr)
    crit = nn.CrossEntropyLoss()

    for _ in range(E):                       # local epochs
        for x, y in loader:                  # minibatches
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            crit(model(x), y).backward()
            opt.step()

    return model   # send weights back to server


def fed_avg(partition, num_rounds=30, C=0.3, E=5, B=32, lr=0.1):
    """
    FedAveraging — Algorithm 1, McMahan et al. 2017.

    w_{t+1} = Σ (n_k / n) * w_k^{t+1}

    Clients do E full epochs locally before syncing.
    This packs more learning into each communication round.
    """
    model = MLP().to(DEVICE)
    n_tot = sum(len(partition[c]) for c in range(NUM_CLIENTS))
    m     = max(1, int(C * NUM_CLIENTS))
    hist  = {'round': [], 'acc': []}

    for rnd in range(1, num_rounds + 1):
        selected = random.sample(range(NUM_CLIENTS), m)

        # Collect updated models from each client
        local_params  = []
        local_weights = []

        for cid in selected:
            idx     = partition[cid]
            updated = client_update(model, idx, E, B, lr)
            local_params.append(get_flat(updated))
            local_weights.append(len(idx))

        # Server: weighted average of client weights
        # w_{t+1} = Σ (n_k / Σn_k) * w_k
        total_w  = sum(local_weights)
        new_flat = sum((w / total_w) * p for w, p in zip(local_weights, local_params))
        set_flat(model, new_flat)

        _, acc = evaluate(model, test_loader)
        hist['round'].append(rnd)
        hist['acc'].append(acc)
        if rnd % 5 == 0:
            print(f"  [FedAvg E={E}] Round {rnd:3d} | Acc: {acc:.1f}%")

    return hist


print("Running FedAvg (E=5)...")
hist_avg_iid    = fed_avg(iid_part,     num_rounds=30, C=0.3, E=5, B=32, lr=0.1)
print("\nNon-IID:")
hist_avg_noniid = fed_avg(non_iid_part, num_rounds=30, C=0.3, E=5, B=32, lr=0.1)

## Step 5 — Ablation: Effect of E (Local Epochs)

The paper's key result: **more local computation → fewer communication rounds**.

We run FedAvg with E=1, E=5, E=20 and compare convergence speed.
This reproduces the intuition behind Table 2 in the paper.

In [ ]:
ablation = {}

for E_val in [1, 5, 20]:
    print(f"Running FedAvg with E={E_val}...")
    ablation[E_val] = fed_avg(
        iid_part, num_rounds=30, C=0.3, E=E_val, B=32, lr=0.1
    )
    print()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("FedAvg vs FedSGD — Replicating McMahan et al. 2017", fontsize=13)

# ── Plot 1: IID ──
ax = axes[0]
ax.plot(hist_sgd_iid['round'],  hist_sgd_iid['acc'],  'r--o', ms=4, lw=2, label='FedSGD')
ax.plot(hist_avg_iid['round'],  hist_avg_iid['acc'],  'b-s',  ms=4, lw=2, label='FedAvg E=5')
ax.set_title("IID Data")
ax.set_xlabel("Communication Rounds")
ax.set_ylabel("Test Accuracy (%)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)

# ── Plot 2: Non-IID ──
ax = axes[1]
ax.plot(hist_sgd_noniid['round'], hist_sgd_noniid['acc'], 'r--o', ms=4, lw=2, label='FedSGD')
ax.plot(hist_avg_noniid['round'], hist_avg_noniid['acc'], 'g-s',  ms=4, lw=2, label='FedAvg E=5')
ax.set_title("Non-IID Data (Pathological)")
ax.set_xlabel("Communication Rounds")
ax.set_ylabel("Test Accuracy (%)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)
ax.text(0.05, 0.1, "Non-IID is harder!\nFedAvg still more robust.",
        transform=ax.transAxes, fontsize=8, color='darkgreen',
        bbox=dict(boxstyle='round', fc='honeydew', ec='green'))

# ── Plot 3: Ablation E ──
ax = axes[2]
colors = ['#e74c3c', '#3498db', '#2ecc71']
for (E_val, hist), color in zip(ablation.items(), colors):
    ax.plot(hist['round'], hist['acc'], color=color, lw=2,
            marker='o', ms=4, label=f'E={E_val}')
ax.set_title("Effect of Local Epochs E (IID)")
ax.set_xlabel("Communication Rounds")
ax.set_ylabel("Test Accuracy (%)")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 100)
ax.text(0.05, 0.05,
        "More E → less communication\nneeded for same accuracy",
        transform=ax.transAxes, fontsize=8, color='navy',
        bbox=dict(boxstyle='round', fc='aliceblue', ec='steelblue'))

plt.tight_layout()
plt.savefig("fedavg_results.png", dpi=130, bbox_inches='tight')
plt.show()

In [ ]:
def rounds_to(hist, target):
    for r, a in zip(hist['round'], hist['acc']):
        if a >= target:
            return r
    return None

TARGET = 80.0
print(f"Rounds to reach {TARGET}% accuracy\n")
print(f"{'Method':<28} {'IID':>8} {'Non-IID':>10}")
print("-" * 48)

r_sgd_iid    = rounds_to(hist_sgd_iid,    TARGET)
r_avg_iid    = rounds_to(hist_avg_iid,    TARGET)
r_sgd_noniid = rounds_to(hist_sgd_noniid, TARGET)
r_avg_noniid = rounds_to(hist_avg_noniid, TARGET)

def fmt(r):
    return str(r) if r else "not reached"

print(f"{'FedSGD':<28} {fmt(r_sgd_iid):>8} {fmt(r_sgd_noniid):>10}")
print(f"{'FedAvg (E=5)':<28} {fmt(r_avg_iid):>8} {fmt(r_avg_noniid):>10}")
print()

if r_sgd_iid and r_avg_iid:
    print(f"Speedup IID    : {r_sgd_iid/r_avg_iid:.1f}×")
if r_sgd_noniid and r_avg_noniid:
    print(f"Speedup Non-IID: {r_sgd_noniid/r_avg_noniid:.1f}×")
print("\nPaper result: 10–100× fewer rounds with FedAvg.")

## What you now understand

### Why FedAvg works
Each round, all clients start from the **same** global model `w_t`.  
After local training they diverge slightly — but since they shared the start,  
averaging their weights lands in a better place than any single client reached.

### The E tradeoff
| E | Communication rounds | Risk |
|---|---|---|
| 1 | Many (like FedSGD) | Safe but slow |
| 5–20 | Far fewer | Sweet spot |
| ∞ | Clients overfit locally | Divergence |

### Privacy (structural, not cryptographic)
Raw data never leaves the device. Only weight deltas travel.  
The paper doesn't add crypto — real deployments combine FedAvg with:
- **Differential Privacy** (Abadi et al. 2016) — formal noise budget
- **Secure Aggregation** (Bonawitz et al. 2016) — server never sees individual updates

### What to read next
- **FedProx** (Li et al. 2020) — handles non-IID better with a proximal term
- **Konečný et al. 2016** — communication compression on top of FedAvg
- **Kairouz et al. 2021** — comprehensive open problems survey